<a href="https://colab.research.google.com/github/ghizlane89/0__GenIA/blob/Bootcamp/W6_D2_DC.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip install -U datasets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 494.8/494.8 kB 14.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.6/193.6 kB 17.4 MB/s eta 0:00:00
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2025.3.2
    Uninstalling fsspec-2025.3.2:
      Successfully uninstalled fsspec-2025.3.2
  Attempting uninstall: datasets
    Found existing installation: datasets 2.14.4
    Uninstalling datasets-2.14.4:
      Successfully uninstalled datasets-2.14.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2025.3.2 requires fsspec==2025.3.2, but you have fsspec 2025.3.0 which is incompatible.
torch 2.6.0+cu124 requires nvidia-cublas-cu12==12.4.5.8; platform_system == "Linux" and platform_machine == "x86_64", but you have nvidia-cublas-cu12 12.5.3.2 which is incompatible.
torch 2.6.0+cu124 requires nvidia-cuda-cupti-cu12==12.4.127; pl

In [1]:
#  2. Load and Inspect Dataset

from datasets import load_dataset
import pandas as pd

raw = load_dataset("ucirvine/sms_spam")
train_ds = raw['train'].select(range(4000))
val_ds = raw['train'].select(range(4000, 5000))

print(train_ds.features)


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


train-00000-of-00001.parquet:   0%|          | 0.00/359k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/5574 [00:00<?, ? examples/s]

{'sms': Value('string'), 'label': ClassLabel(names=['ham', 'spam'])}


In [2]:
#  1. Installation
!pip install --quiet evaluate transformers[sentencepiece]

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.7 MB/s eta 0:00:00


In [3]:
# ✅ 3. Tokenization
from transformers import GPT2Tokenizer

model_name = "gpt2"
tokenizer = GPT2Tokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

def tokenize_fn(examples):
    return tokenizer(
        examples["sms"],
        padding="max_length",
        truncation=True,
        max_length=64
    )

train_tok = train_ds.map(tokenize_fn, batched=True)
val_tok = val_ds.map(tokenize_fn, batched=True)

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

Map:   0%|          | 0/4000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

In [4]:
# ✅ 4. Model Initialization
import torch
from transformers import GPT2ForSequenceClassification

model = GPT2ForSequenceClassification.from_pretrained(
    model_name,
    num_labels=2,
    pad_token_id=tokenizer.eos_token_id
)


model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

Some weights of GPT2ForSequenceClassification were not initialized from the model checkpoint at gpt2 and are newly initialized: ['score.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [6]:

# ✅ 5. Metrics Definition
import evaluate
import numpy as np

accuracy = evaluate.load("accuracy")
precision = evaluate.load("precision")
recall = evaluate.load("recall")
f1 = evaluate.load("f1")

def compute_metrics(pred):
    logits, labels = pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy":  accuracy.compute(predictions=preds, references=labels)["accuracy"],
        "precision": precision.compute(predictions=preds, references=labels)["precision"],
        "recall":    recall.compute(predictions=preds, references=labels)["recall"],
        "f1":        f1.compute(predictions=preds, references=labels)["f1"]
    }

Dans un dataset déséquilibré comme les SMS spam (souvent plus de “ham” que de “spam”), pourquoi est-il important de suivre la précision (precision) et le rappel (recall) en plus de l'exactitude (accuracy) ?
Parce que l’accuracy peut être trompeuse.

Exemple :

90% des messages sont “ham”.

Si ton modèle prédit toujours “ham”, il aura 90% d’accuracy… mais il n’a détecté aucun spam.

 Donc :

Précision (precision) : parmi les messages que le modèle a classés “spam”, combien sont vraiment du spam ? ( utile pour éviter les faux positifs)






Rappel (recall) : parmi les vrais spams, combien ont été détectés par le modèle ? ( utile pour éviter les faux négatifs)

F1-score : équilibre entre précision et rappel.

Comment interpréter un modèle qui a une haute accuracy mais un faible recall sur le spam ?
Cela veut dire que :

Le modèle fonctionne bien sur les “ham” (messages normaux).

Mais il rate beaucoup de spams (faux négatifs).

Il est trop conservateur : il préfère ne rien signaler plutôt que de se tromper.

 C’est dangereux si l’objectif est de filtrer les spams, car tu risques de laisser passer beaucoup de messages indésirables.

In [7]:
# ✅ 6. TrainingArguments
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./gpt2-sms",
    do_train=True,
    do_eval=True,
    eval_steps=500,
    save_steps=500,
    logging_dir="./logs",
    logging_steps=500,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    num_train_epochs=3,
    learning_rate=5e-5,
    weight_decay=0.01,
    report_to=None,
    save_total_limit=1,
)

Quel est l'effet du weight_decay pendant le fine-tuning ? Quand choisir une valeur plus haute ou plus basse ?

Le weight_decay correspond à une régularisation de type L2 sur les poids du modèle.

Son but :
Limiter l'amplitude des poids pour éviter qu'ils ne deviennent trop grands, ce qui réduit le risque de surapprentissage (overfitting).

Quand choisir une certaine valeur :

Pour mieux généraliser sur des données réelles : on utilise une valeur modérée, par exemple 0.01.

Pour éviter l’overfitting sur un petit dataset : on peut choisir une valeur plus élevée, comme 0.05.

Pour ne pas trop perturber l’apprentissage tout en régularisant légèrement : on choisira une valeur plus faible, comme 0.001 ou 0.0001.

À noter :

Un weight_decay trop élevé peut empêcher le modèle d’apprendre correctement.

Un weight_decay trop faible peut entraîner un surapprentissage, surtout si le dataset est petit ou bruité.



In [8]:
# ✅ 7. Trainer Setup, Training & Final Evaluation

from transformers import TrainingArguments, Trainer

# Configuration des arguments d'entraînement
training_args = TrainingArguments(
    output_dir="./gpt2-sms",
    do_train=True,
    do_eval=True,
    eval_steps=500,
    save_steps=500,
    logging_dir="./logs",
    logging_steps=500,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    num_train_epochs=3,
    learning_rate=5e-5,
    weight_decay=0.01,
    report_to="none",  # <== Désactive WandB
    save_total_limit=1,
)

# Création de l'entraîneur
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_tok,
    eval_dataset=val_tok,
    compute_metrics=compute_metrics,
)

# Lancement de l'entraînement
trainer.train()

# Évaluation finale
metrics = trainer.evaluate()
print(metrics)


Step,Training Loss
500,0.234300
1000,0.134000
1500,0.064200
2000,0.049600
2500,0.023400
3000,0.030200


{'eval_loss': 0.06166849657893181, 'eval_accuracy': 0.99, 'eval_precision': 0.9849624060150376, 'eval_recall': 0.9424460431654677, 'eval_f1': 0.9632352941176471, 'eval_runtime': 4.1694, 'eval_samples_per_second': 239.841, 'eval_steps_per_second': 59.96, 'epoch': 3.0}


Le modèle fine-tuné sur la classification des SMS a obtenu les résultats suivants sur le jeu de validation :

Accuracy : 99 %

Précision : 98,5 %

Rappel : 94,2 %

F1-score : 96,3 %

Loss : 0,0617

Ces résultats montrent que le modèle est à la fois précis et fiable. Il détecte efficacement les messages spam sans générer beaucoup de fausses alertes. Le score F1 élevé confirme un bon équilibre entre détection des spams et respect des messages normaux.

Le modèle est donc bien entraîné et généralise correctement sur des données nouvelles. Il peut être utilisé en pratique pour filtrer des SMS avec une grande efficacité.

